# 04 — Score Matching & SDEs

Diffusion models connect to stochastic differential equations and score-based generative modelling.

## Score Functions and Stochastic Differential Equations

**Score function**: the gradient of the log-density with respect to the data:
$$s(x) = \nabla_x \log p(x)$$
The score points in the direction of increasing probability density — it is a vector field over data space.

**Score matching** (Hyvärinen 2005) learns the score function without needing samples from the normalised density:
$$\mathcal{L}_{SM} = \mathbb{E}_{p(x)}\left[\text{tr}(\nabla_x s_\theta(x)) + \frac{1}{2}||s_\theta(x)||^2\right]$$

**Denoising score matching** (Vincent 2011) is equivalent but more tractable: add noise $\tilde{x} = x + \sigma\epsilon$ and train:
$$\mathcal{L}_{DSM} = \mathbb{E}\left[||s_\theta(\tilde{x}, \sigma) - \nabla_{\tilde{x}} \log p(\tilde{x}|x)||^2\right]$$

**SDE formulation** (Song et al., 2021): the forward process is a continuous-time SDE:
$$dx = f(x,t)dt + g(t)dW$$
For DDPM, $f(x,t) = -\frac{1}{2}\beta(t)x$, $g(t) = \sqrt{\beta(t)}$.

The reverse SDE uses the score function:
$$dx = [f(x,t) - g(t)^2 \nabla_x \log p_t(x)]dt + g(t)d\bar{W}$$
Since $\epsilon_\theta \approx -\sigma_t s_\theta$, the noise prediction network and score network are equivalent.

In [ ]:
# Score-based model: learning the score function directly
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# 2D dataset — two Gaussians
def sample_data(n=1000):
    half = n // 2
    x1 = torch.randn(half, 2) * 0.3 + torch.tensor([1.5, 1.5])
    x2 = torch.randn(half, 2) * 0.3 + torch.tensor([-1.5, -1.5])
    return torch.cat([x1, x2])

data = sample_data(2000)

# Score network
class ScoreNet(nn.Module):
    def __init__(self, sigma_min=0.01, sigma_max=3.0, n_sigmas=10):
        super().__init__()
        self.sigmas = torch.exp(torch.linspace(np.log(sigma_min), np.log(sigma_max), n_sigmas))
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )

    def forward(self, x, sigma_idx):
        sigma = self.sigmas[sigma_idx].unsqueeze(1)  # (batch, 1)
        sigma_feat = torch.log(sigma)  # log-sigma as extra feature
        h = torch.cat([x, sigma_feat], dim=-1)
        score = self.net(h) / sigma  # score = -perturbed_noise / sigma^2
        return score

score_net = ScoreNet()
opt = torch.optim.Adam(score_net.parameters(), lr=1e-3)

# Denoising score matching
for step in range(2000):
    idx = torch.randint(0, len(data), (256,))
    x0 = data[idx]
    sigma_idx = torch.randint(0, len(score_net.sigmas), (256,))
    sigma = score_net.sigmas[sigma_idx].unsqueeze(1)
    noise = torch.randn_like(x0)
    x_noisy = x0 + sigma * noise
    target_score = -noise / sigma
    predicted_score = score_net(x_noisy, sigma_idx)
    loss = (sigma**2 * (predicted_score - target_score)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

print('Score network trained')

In [ ]:
# Langevin dynamics sampling using the learned score
@torch.no_grad()
def langevin_sample(score_net, n_samples, n_steps=1000, step_size=0.01,
                    sigma_idx=0):
    """Annealed Langevin dynamics: run at each noise level."""
    x = torch.randn(n_samples, 2) * 3.0

    for si in range(len(score_net.sigmas) - 1, -1, -1):
        si_batch = torch.full((n_samples,), si)
        sigma = score_net.sigmas[si].item()
        alpha = step_size * sigma**2

        for _ in range(n_steps // len(score_net.sigmas)):
            score = score_net(x, si_batch)
            noise = torch.randn_like(x)
            x = x + alpha * score + (2 * alpha)**0.5 * noise

    return x

samples = langevin_sample(score_net, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(data[:200, 0], data[:200, 1], alpha=0.3, s=5)
ax1.set_title('Real data')
ax1.set_xlim(-4, 4); ax1.set_ylim(-4, 4)
ax2.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), alpha=0.3, s=5, color='orange')
ax2.set_title('Langevin samples')
ax2.set_xlim(-4, 4); ax2.set_ylim(-4, 4)
plt.suptitle('Score-Based Generation via Annealed Langevin Dynamics')
plt.tight_layout()
plt.savefig('/tmp/score_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Score-based sampling complete')